# installing RDkit

In [ ]:
conda install conda-forge::rdkit

In [ ]:
pip install torch-scatter


In [ ]:
pip install torch-sparse 


In [ ]:
pip install torch-cluster

# Working

In [24]:
pip --version torch

pip 23.3.1 from /Applications/anaconda3/lib/python3.11/site-packages/pip (python 3.11)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
print(torch.__version__)

In [ ]:
print(torch.backends.mps.is_available())

In [ ]:
import torchvision
torchvision.__version__

In [ ]:
import rdkit
print(rdkit.__version__)

In [26]:
from torch_geometric.datasets import MoleculeNet
 
# Load the ESOL dataset
data = MoleculeNet(root=".", name="ESOL")

ModuleNotFoundError: No module named 'torch_geometric.datasets.planetoid'

In [ ]:
data

In [ ]:
print("Dataset type: ", type(data))
print("Dataset features: ", data.num_features)
print("Dataset target: ", data.num_classes)
print("Dataset length: ", data.len)
print("Dataset sample: ", data[0])
print("Sample  nodes: ", data[0].num_nodes)
print("Sample  edges: ", data[0].num_edges)

In [ ]:
data[0].x

In [ ]:
data[0].y

# IIST GCN

In [28]:
import torch
import torch.nn.functional as F
import torch.optim as optim
print('success')

success


In [ ]:
import numpy as np
my_matrix = torch.tensor([[0,0,1,0],[5,0,0,0],[0,2,0,0],[0,0,0,4]])
my_matrix

In [ ]:
my_matrix[:, 1:-1]

In [ ]:
my_matrix.to_sparse_csr()

In torch, csr matrix is called on the instance

In [30]:
from torch_geometric.transforms import NormalizeFeatures

In [ ]:
dataset = Planetoid('/Users/poorvaichandrasen/GCN_IIST/data', name = 'Cora', transform = NormalizeFeatures())

In [ ]:
print(dataset[0])

In [ ]:
print(len(dataset))
print(dataset)

In [ ]:
dir(dataset)

In [ ]:
print(dataset.num_classes)
print(dataset.num_features)
print(dataset.num_node_features)
print(dataset.num_edge_features)
print

In [ ]:
data = dataset[0]
dir(data)

In [ ]:
print(data.num_nodes)
print(data.num_edges)

In [ ]:
data.train_mask.sum()

In [ ]:
len(data.train_mask)== data.num_nodes

In [ ]:
vars(data.train_mask)

In [ ]:
class Dog():
    def __init__(self, name, age):
        self.name = name
        self.age = age
    def bark (self, word):
        print(self.name, 'says', word)

In [ ]:
Obj1 = Dog('blackie', 2)

In [ ]:
Obj1.bark('hello')

# Creating model

To define your model, you need to define its attributes and its forward function. 
<br>
#### Model attributes:
1. define its self.layers with their attributes (no of features/classes etc.)
<br>
<br>
Forward function: pass matrix to the layers, store relu input, dropout, pass to second layer, store its softmax
<br><Br>
#### Define layer class (attributes):
1. define its attributes as self weights, bias, and input/output size and initialise parameters
2. define its forward function
3. define its representation
4. define how to initialise parameters

In [6]:
#Create model
import torch.nn as nn
import numpy as np

In [4]:
import torch.nn.functional as F
from torch.nn import Parameter

In [8]:
class myGCNmodel(nn.Module):
    def __init__(self, num_feat, hdn_units, num_class):
        super().__init__()
        self.layer1 = myGCNLayer(num_feat, hdn_units) 
        #Do not keep layers as local variables, they are attributes of the instance
        self.layer2 = myGCNLayer(hdn_units, num_class)
        

    def forward(self, feature_mat, adj_mat): #Do not forget self
        x = F.relu(self.layer1.layer_forward (feature_mat, adj_mat))
        #x = F.dropout(x, self.dropout, training=self.training)
        x = self.layer2.layer_forward (x, adj_mat)
        return F.log_softmax(x, dim=1)
    

In [12]:
class myGCNLayer(nn.Module):
    def __init__(self, in_size, out_size, bias = True):
        super().__init__()
        self.in_size = in_size
        self.out_size = out_size
        self.weights = Parameter(torch.FloatTensor(self.in_size, self.out_size))
        if bias:
            self.bias = Parameter(torch.FloatTensor(self.out_size))  #keep bias as row vector
        else:
            self.register_parameter('bias', None)
        self.initialise_parameters()
    

    def initialise_parameters(self):
        torch.manual_seed(42)
        self.weights = Parameter(torch.rand(self.in_size, self.out_size))
        if self.bias is not None:
            self.bias = Parameter(torch.rand(self.out_size))
            
    def layer_forward(self, in_features, adj):
        support = torch.mm(in_features, self.weights)
        output = torch.spmm(adj, support)
        if self.bias is not None:
            return output + self.bias
        else:
            return output
            
    def __repr__(self):
        return self.__class__.__name__ + ' (' \
               + str(self.in_size) + ' -> ' \
               + str(self.out_size) + ')'

In [ ]:
#train
model = myGCNmodel(1433, 16, 7)

In [ ]:
model

# Prepare dataset

We require:
1. Feature matrix: In cora, it is named as x
2. adjacency matrix as sparse
3. classes?: In cora, it is named as y

In [6]:
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures

In [20]:
dataset = Planetoid('/Users/poorvaichandrasen/GCN_IIST/data', name = 'Cora', transform = NormalizeFeatures())
data = dataset[0]

ModuleNotFoundError: No module named 'torch_geometric.datasets.planetoid'

In [ ]:
# the adjacency matrix is not of the shape required
#change it to sparse index

from torch_sparse import SparseTensor
edge_index = SparseTensor(row=data.edge_index[0], col=data.edge_index[1], sparse_sizes=(data.num_nodes, data.num_nodes))


In [ ]:
def encode_onehot(labels):
    classes = set(labels)
    classes_dict = {c: np.identity(len(classes))[i, :] for i, c in
                    enumerate(classes)}
    labels_onehot = np.array(list(map(classes_dict.get, labels)),
                             dtype=np.int32)
    return labels_onehot

In [ ]:
data.classes

# Training

In [ ]:
model = myGCNmodel(data.num_features, 16, dataset.num_classes)
print(model)

In [ ]:
device = torch.device("cpu")
model = model.to(device)
data = data.to(device)

In [ ]:
learning_rate = 0.01
decay = 5e-4
optimizer = torch.optim.Adam(model.parameters(), 
                             lr=learning_rate, 
                             weight_decay=decay)
# Define loss function (CrossEntropyLoss for Classification Problems with 
# probability distributions)
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
def train():
      model.train()
      optimizer.zero_grad() 
      # Use all data as input, because all nodes have node features
      out = model.forward(data.x, edge_index)  
      # Only use nodes with labels available for loss calculation --> mask
      loss = criterion(out[data.train_mask], data.y[data.train_mask])  
      loss.backward() 
      optimizer.step()
      return loss
    
def test():
      model.eval()
      out = model(data.x, data.edge_index)
      # Use the class with highest probability.
      pred = out.argmax(dim=1)  
      # Check against ground-truth labels.
      test_correct = pred[data.test_mask] == data.y[data.test_mask]  
      # Derive ratio of correct predictions.
      test_acc = int(test_correct.sum()) / int(data.test_mask.sum())  
      return test_acc

losses = []
for epoch in range(0, 1001):
    loss = train()
    losses.append(loss)
    if epoch % 100 == 0:
      print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

In [ ]:
data.x.shape

In [ ]:
type(data.edge_index)